# 6. Evaluación y regularización del modelo CNN

Este notebook se utilizará para:

1. Cargar y comprender `X.npy` y `Y.npy`.
2. Corregir la división en entrenamiento, validación y prueba.
3. Construir un modelo base sin regularización.
4. Comparar modelos de distinta complejidad y con L1, L2 y Dropout.
5. Seleccionar el mejor modelo utilizando validación.
6. Evaluar una sola vez el modelo final con el conjunto de prueba.

> El notebook original del grupo se conserva como respaldo y no se modifica.

## 1. Conectar Google Drive

Colab ejecuta el código en una máquina temporal. Por eso los datos, modelos y resultados importantes se guardarán en Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Definir las carpetas del proyecto

La siguiente celda crea una estructura persistente dentro de Google Drive. Los archivos `X.npy` y `Y.npy` deben copiarse manualmente a la carpeta `data`.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Taller3_Sign_Language')
DATA_DIR = DRIVE_ROOT / 'data'
MODELS_DIR = DRIVE_ROOT / 'models'
LOGS_DIR = DRIVE_ROOT / 'logs'
OUTPUTS_DIR = DRIVE_ROOT / 'outputs'

for carpeta in [DATA_DIR, MODELS_DIR, LOGS_DIR, OUTPUTS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print('Carpeta principal:', DRIVE_ROOT)
print('Datos:', DATA_DIR)
print('Modelos:', MODELS_DIR)

## 3. ¿Qué son X e Y?

- **X** contiene las entradas del modelo: las imágenes.
- **Y** contiene las respuestas correctas: la clase asociada a cada imagen.

`X[i]` y `Y[i]` forman una pareja. Es decir, `Y[i]` indica qué dígito aparece en `X[i]`.

Los archivos `.npy` son archivos binarios de NumPy. No están pensados para abrirse con Bloc de notas; se leen mediante `numpy.load()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_PATH = DATA_DIR / 'X.npy'
Y_PATH = DATA_DIR / 'Y.npy'

if not X_PATH.exists() or not Y_PATH.exists():
    raise FileNotFoundError(
        'Faltan X.npy o Y.npy. Copia ambos archivos a: ' + str(DATA_DIR)
    )

X = np.load(X_PATH)
Y_raw = np.load(Y_PATH)

print('Forma de X:', X.shape)
print('Tipo de X:', X.dtype)
print('Rango de X:', X.min(), 'a', X.max())
print('Forma de Y:', Y_raw.shape)
print('Tipo de Y:', Y_raw.dtype)

### Interpretación de las formas

- `X.shape = (2062, 64, 64)` significa: 2062 imágenes, cada una de 64 × 64 píxeles.
- `Y.shape = (2062, 10)` significa: 2062 etiquetas, con 10 posiciones posibles, una por clase.

La etiqueta usa **one-hot encoding**. Por ejemplo:

```text
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
```

indica que la posición 2 es la clase activa.

In [ ]:
indice = 0

plt.figure(figsize=(4, 4))
plt.imshow(X[indice], cmap='gray')
plt.title(f'Imagen X[{indice}]')
plt.axis('off')
plt.show()

print('Etiqueta original Y_raw[0]:', Y_raw[indice])
print('Posición activa:', np.argmax(Y_raw[indice]))

## 4. Reordenar las columnas de Y

El trabajo anterior utilizó el siguiente orden para asociar las columnas originales con los dígitos 0–9. Se conservará de forma explícita y posteriormente se verificará visualmente.

In [ ]:
ORDEN_CLASES = [1, 4, 8, 7, 6, 9, 3, 2, 5, 0]
Y = Y_raw[:, ORDEN_CLASES]
labels = np.argmax(Y, axis=1)

print('Etiqueta reordenada Y[0]:', Y[0])
print('Dígito asignado a X[0]:', labels[0])
print('Cantidad por clase:', np.bincount(labels))

## 5. Verificar visualmente una imagen por clase

Esta comprobación es importante: la imagen mostrada debe coincidir con el título del dígito. Si no coincide, no se debe continuar con el entrenamiento.

In [ ]:
plt.figure(figsize=(12, 5))

for digito in range(10):
    idx = np.where(labels == digito)[0][0]
    plt.subplot(2, 5, digito + 1)
    plt.imshow(X[idx], cmap='gray')
    plt.title(f'Dígito: {digito}')
    plt.axis('off')

plt.tight_layout()
plt.show()

## 6. Preparar las imágenes y realizar una división estratificada

La división será:

- 80 % entrenamiento.
- 10 % validación.
- 10 % prueba.

La opción `stratify` mantiene aproximadamente la misma proporción de cada dígito en los tres conjuntos.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# X.npy ya está normalizado aproximadamente en el rango [0, 1].
X_cnn = X.astype('float32').reshape(-1, 64, 64, 1)

(
    X_train, X_temp,
    y_train, y_temp,
    labels_train, labels_temp
) = train_test_split(
    X_cnn, Y, labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

(
    X_val, X_test,
    y_val, y_test,
    labels_val, labels_test
) = train_test_split(
    X_temp, y_temp, labels_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=labels_temp
)

print('Entrenamiento:', X_train.shape, np.bincount(labels_train))
print('Validación:', X_val.shape, np.bincount(labels_val))
print('Prueba:', X_test.shape, np.bincount(labels_test))

## 7. Regla de trabajo para el punto 6

Desde este momento:

- El conjunto de **entrenamiento** se usa para ajustar los pesos.
- El conjunto de **validación** se usa para comparar modelos e hiperparámetros.
- El conjunto de **prueba** no se usa para tomar decisiones y se evalúa solamente al final.

La siguiente etapa será construir el **Modelo A: CNN base sin regularización**.